# Complete DeepLog LSTM Log Anomaly Detection Guide
**From absolute zero → production metrics**

This notebook is the single source of truth for the `lstm-log-analysis` project.

## 0. Environment & Reproducibility

In [ ]:
# ============================================================
# CELL 0 – installs & seeds (run once)
# ============================================================
import os, sys, json, math, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

## 1. Core Terminology & Mental Model

In [ ]:
# ============================================================
# CELL 1 – quick glossary as Python dict (for later reference)
# ============================================================
TERMINOLOGY = {
    'log_template': 'A pattern with variables replaced by <*> or *',
    'event_id':     'Integer ID assigned to each unique template',
    'session':      'Ordered list of event_ids that belong to one task',
    'window_h':     'How many past events the LSTM sees (hyper-parameter h)',
    'top_k':        'If true next event is among the k most probable → normal',
    'anomaly_score':'Fraction of windows in a session that were top-k misses',
    'stacked_lstm': 'Two LSTM layers → learns short + longer grammar'
}
for k, v in TERMINOLOGY.items():
    print(f'{k:15s} : {v}')

## 2. Synthetic Data Generator (so you can run everything offline)

In [ ]:
# ============================================================
# CELL 2 – tiny but realistic log generator
# Mimics the real dataset.py parsing step
# ============================================================
def generate_synthetic_sessions(n_normal=300, n_anomaly=200, max_len=50, vocab_size=20):
    """
    Returns
    -------
    sessions : list of list[int]   # each inner list is one session
    labels   : list[int]           # 0 = normal, 1 = anomaly
    """
    sessions, labels = [], []

    # ----- normal grammar: almost always 1→2→3→4→1… with small noise -----
    normal_pattern = [1, 2, 3, 4, 5, 6, 7, 8]
    for _ in range(n_normal):
        length = np.random.randint(15, max_len)
        sess = [normal_pattern[i % len(normal_pattern)] for i in range(length)]
        # tiny noise
        for i in range(length):
            if np.random.rand() < 0.05:
                sess[i] = np.random.randint(1, vocab_size)
        sessions.append(sess)
        labels.append(0)

    # ----- anomalous: sudden jump, out-of-order, or rare event burst -----
    for _ in range(n_anomaly):
        length = np.random.randint(15, max_len)
        sess = [normal_pattern[i % len(normal_pattern)] for i in range(length)]
        # inject structural fault
        fault_type = np.random.choice(['jump', 'burst', 'reverse'])
        pos = np.random.randint(5, length-5)
        if fault_type == 'jump':
            sess[pos] = 19          # very rare event
        elif fault_type == 'burst':
            sess[pos:pos+3] = [18, 18, 18]
        else:
            sess[pos:pos+4] = list(reversed(sess[pos:pos+4]))
        sessions.append(sess)
        labels.append(1)

    return sessions, labels

sessions, labels = generate_synthetic_sessions()
print(f'Generated {len(sessions)} sessions  (normal={labels.count(0)}, anomaly={labels.count(1)})')
print('Example normal session:', sessions[0][:20])
print('Example anomaly session:', sessions[-1][:20])

## 3. Sliding-Window Feature Engineering

In [ ]:
# ============================================================
# CELL 3 – exactly what src/features.py does
# ============================================================
def build_windows(sessions, labels, h=10):
    """
    From each session produce overlapping windows.

    X[i] = [e_{t-h}, e_{t-h+1}, ..., e_{t-1}]
    y[i] = e_t
    session_ids[i] = which session this window came from
    """
    X, y, session_ids, window_labels = [], [], [], []
    for s_idx, (sess, lab) in enumerate(zip(sessions, labels)):
        if len(sess) <= h:
            continue
        for i in range(len(sess) - h):
            X.append(sess[i : i+h])
            y.append(sess[i+h])
            session_ids.append(s_idx)
            window_labels.append(lab)          # inherit session label
    return np.array(X), np.array(y), np.array(session_ids), np.array(window_labels)

H = 10          # history window (same as README $h=10$)
X, y, sess_ids, win_labels = build_windows(sessions, labels, h=H)
print('X shape (windows, h):', X.shape)
print('y shape:', y.shape)
print('Unique events (vocab):', np.unique(y).max() + 1)

## 4. Train / Val / Test Split (zero leakage)

In [ ]:
# ============================================================
# CELL 4 – split BY SESSION, never by window (prevents leakage)
# ============================================================
from sklearn.model_selection import train_test_split

unique_sessions = np.unique(sess_ids)
session_label_map = {s: labels[s] for s in unique_sessions}

# We only train on NORMAL sessions (DeepLog philosophy)
normal_sessions = [s for s in unique_sessions if session_label_map[s] == 0]
anomaly_sessions = [s for s in unique_sessions if session_label_map[s] == 1]

train_sess, temp_sess = train_test_split(normal_sessions, test_size=0.3, random_state=SEED)
val_sess, test_normal_sess = train_test_split(temp_sess, test_size=0.5, random_state=SEED)

# Test set = held-out normal + all anomalies
test_sess = list(test_normal_sess) + anomaly_sessions

def mask_windows(session_list):
    mask = np.isin(sess_ids, session_list)
    return X[mask], y[mask], sess_ids[mask], win_labels[mask]

X_train, y_train, _, _ = mask_windows(train_sess)
X_val,   y_val,   _, _ = mask_windows(val_sess)
X_test,  y_test,  sid_test, lab_test = mask_windows(test_sess)

print(f'Train windows: {len(X_train)} (only normal)')
print(f'Val   windows: {len(X_val)}')
print(f'Test  windows: {len(X_test)} (normal+anomaly)')

## 5. Build the Stacked LSTM (exactly modeling/train.py)

In [ ]:
# ============================================================
# CELL 5 – model architecture with line-by-line comments
# ============================================================
VOCAB_SIZE = int(max(y.max(), X.max()) + 1)   # +1 because IDs start at 0 or 1
EMBED_DIM  = 64
LSTM_UNITS = 64
DROPOUT    = 0.1

def build_deeplog_model(vocab_size, h, embed_dim=64, lstm_units=64):
    inp = keras.Input(shape=(h,), name='event_history')          # (batch, h)

    # 1. Turn each event ID into a dense vector
    x = layers.Embedding(input_dim=vocab_size,
                         output_dim=embed_dim,
                         name='event_embedding')(inp)           # (batch, h, embed)

    # 2. First LSTM layer (returns sequence for stacking)
    x = layers.LSTM(lstm_units, return_sequences=True,
                    dropout=DROPOUT, name='lstm_1')(x)          # (batch, h, units)

    # 3. Second LSTM layer (returns only last state)
    x = layers.LSTM(lstm_units, return_sequences=False,
                    dropout=DROPOUT, name='lstm_2')(x)          # (batch, units)

    # 4. Softmax over every possible next event
    out = layers.Dense(vocab_size, activation='softmax',
                       name='next_event_probs')(x)              # (batch, vocab)

    model = keras.Model(inp, out, name='DeepLog_StackedLSTM')
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_deeplog_model(VOCAB_SIZE, H, EMBED_DIM, LSTM_UNITS)
model.summary()

## 6. Train

In [ ]:
# ============================================================
# CELL 6 – training with EarlyStopping & model checkpoint
# ============================================================
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint('models/lstm_log_anomaly_model.keras',
                                    monitor='val_loss', save_best_only=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# save history for reports/
with open('models/training_history.json', 'w') as f:
    json.dump({k: [float(x) for x in v] for k, v in history.history.items()}, f)

## 7. Top-K Inference & Session Anomaly Scoring
(This is the heart of `modeling/predict.py`)

In [ ]:
# ============================================================
# CELL 7 – vectorized Top-K anomaly detection
# ============================================================
def predict_topk_flags(model, X, y_true, k=9):
    """
    Returns boolean array: True = this window is anomalous
    (i.e. true next event not in model top-k)
    """
    probs = model.predict(X, batch_size=512, verbose=0)        # (N, vocab)
    # fast top-k using argpartition
    topk_idx = np.argpartition(probs, -k, axis=1)[:, -k:]      # (N, k)
    # check whether y_true is inside each row’s top-k
    flags = np.array([y_true[i] not in topk_idx[i] for i in range(len(y_true))])
    return flags

def session_anomaly_scores(flags, session_ids):
    """Aggregate window flags → one score per session."""
    df = pd.DataFrame({'sid': session_ids, 'flag': flags.astype(int)})
    scores = df.groupby('sid')['flag'].mean().to_dict()        # fraction
    return scores

# ----- run on test set -----
K = 9
window_flags = predict_topk_flags(model, X_test, y_test, k=K)
sess_scores  = session_anomaly_scores(window_flags, sid_test)

# build final results list (one dict per test session)
results = []
for s in np.unique(sid_test):
    results.append({
        'session_id': int(s),
        'true_label': int(session_label_map[s]),
        'anomaly_score': float(sess_scores[s])
    })

print(f'Computed scores for {len(results)} test sessions')
print('Example:', results[:3])

## 8. Evaluation Metrics (Precision, Recall, F1, ROC-AUC, PR-AUC)

In [ ]:
# ============================================================
# CELL 8 – full metric suite (matches README table)
# ============================================================
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             confusion_matrix, roc_curve, precision_recall_curve)

y_true = np.array([r['true_label'] for r in results])
y_score = np.array([r['anomaly_score'] for r in results])

# You can sweep threshold; 0.5 is a common start
THRESHOLD = 0.5
y_pred = (y_score >= THRESHOLD).astype(int)

prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)
roc  = roc_auc_score(y_true, y_score)
pr   = average_precision_score(y_true, y_score)

print('='*50)
print(f'Precision : {prec*100:.2f}%')
print(f'Recall    : {rec*100:.2f}%')
print(f'F1-Score  : {f1:.4f}')
print(f'ROC-AUC   : {roc:.4f}')
print(f'PR-AUC    : {pr:.4f}')
print('='*50)

cm = confusion_matrix(y_true, y_pred)
print('Confusion matrix:\n', cm)

## 9. Visualizations (publication quality)

In [ ]:
# ============================================================
# VISUALIZATION 1: Training curves
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['loss'], label='train')
ax[0].plot(history.history['val_loss'], label='val')
ax[0].set_title('Cross-Entropy Loss')
ax[0].legend(); ax[0].grid(True, alpha=0.3)

ax[1].plot(history.history['accuracy'], label='train')
ax[1].plot(history.history['val_accuracy'], label='val')
ax[1].set_title('Next-Event Accuracy')
ax[1].legend(); ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('reports/figures/training_curves.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# VISUALIZATION 2: ROC & PR curves
# ============================================================
fpr, tpr, _ = roc_curve(y_true, y_score)
prec_c, rec_c, _ = precision_recall_curve(y_true, y_score)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(fpr, tpr, color='#2196F3', lw=2, label=f'AUC={roc:.3f}')
ax[0].plot([0,1],[0,1],'k--')
ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].set_title('ROC Curve')
ax[0].legend(); ax[0].grid(True, alpha=0.3)

ax[1].plot(rec_c, prec_c, color='#FF9800', lw=2, label=f'AP={pr:.3f}')
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision'); ax[1].set_title('Precision-Recall')
ax[1].legend(); ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('reports/figures/roc_pr_curves.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# VISUALIZATION 3: Anomaly Score Distribution
# ============================================================
# Shows how well the model separates normal from anomaly.
# Good model: two distinct, non-overlapping distributions.

normal_scores = [r['anomaly_score'] for r in results if r['true_label'] == 0]
anomaly_scores_list = [r['anomaly_score'] for r in results if r['true_label'] == 1]

fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(normal_scores, bins=30, alpha=0.6, color='#4CAF50',
        label=f'Normal (n={len(normal_scores)})', density=True)
ax.hist(anomaly_scores_list, bins=30, alpha=0.6, color='#F44336',
        label=f'Anomaly (n={len(anomaly_scores_list)})', density=True)

ax.set_xlabel('Anomaly Score (fraction of anomalous windows)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Anomaly Score Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reports/figures/anomaly_score_distribution.png', dpi=150)
plt.show()

print(f'\nNormal sessions:  mean score = {np.mean(normal_scores):.4f}, '
      f'std = {np.std(normal_scores):.4f}')
print(f'Anomaly sessions: mean score = {np.mean(anomaly_scores_list):.4f}, '
      f'std = {np.std(anomaly_scores_list):.4f}')

In [ ]:
# ============================================================
# VISUALIZATION 4: Top-K Sensitivity Trade-off
# ============================================================
ks = [1, 3, 5, 9, 15]
rows = []
for k in ks:
    flags_k = predict_topk_flags(model, X_test, y_test, k=k)
    scores_k = session_anomaly_scores(flags_k, sid_test)
    y_sc = np.array([scores_k[s] for s in np.unique(sid_test)])
    y_pr = (y_sc >= 0.5).astype(int)
    rows.append({
        'k': k,
        'precision': precision_score(y_true, y_pr, zero_division=0),
        'recall':    recall_score(y_true, y_pr, zero_division=0),
        'f1':        f1_score(y_true, y_pr, zero_division=0)
    })

df_k = pd.DataFrame(rows)
print(df_k.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df_k['k'], df_k['precision'], 'o-', label='Precision')
ax.plot(df_k['k'], df_k['recall'],    's-', label='Recall')
ax.plot(df_k['k'], df_k['f1'],        'D-', label='F1')
ax.set_xlabel('Top-K'); ax.set_title('Top-K Sensitivity Trade-off')
ax.legend(); ax.grid(True, alpha=0.3)
plt.savefig('reports/figures/top_k_sensitivity.png', dpi=150)
plt.show()

## 10. How to Use the Real Project Codebase

In [ ]:
# ============================================================
# CELL 10 – mapping notebook → real source files
# ============================================================
print("""
Real file                ↔  What we did in this notebook
---------------------------------------------------------------
src/config.py              paths, h, k, vocab, seeds
src/dataset.py             raw log → templates → sessions (Cell 2)
src/features.py            sliding windows + zero-leakage split (Cells 3-4)
src/modeling/train.py      build + fit LSTM (Cells 5-6)
src/modeling/predict.py    top-k flags + session scores (Cell 7)
src/plots.py               all figures (Cells 9.x)
src/test_scripts/          end-to-end integration test
tests/                     unit tests for every module
Makefile                   make data / make train / make predict / make test
""")

## 11. Production Checklist & Next Steps

In [ ]:
print("""
✓ 1. Replace synthetic generator with real Drain/regex parser on your logs
✓ 2. Put normal-only sessions into data/processed/train
✓ 3. make train          → models/lstm_log_anomaly_model.keras
✓ 4. make predict        → reports/evaluation_metrics.json
✓ 5. Tune h ∈ {5…20} and k ∈ {1…15} on validation
✓ 6. Add real-time streaming (Kafka → predict.py → alert)
✓ 7. Monitor drift: retrain when top-k accuracy on recent normal data drops
""")

## 12. Key Hyper-parameters (cheat sheet)

In [ ]:
CHEATSHEET = {
    'h (history window)': '10 (README default). Larger → more context, slower',
    'k (top-k)':          '9 gives zero FPs on the paper’s HDFS; lower k → higher recall',
    'LSTM units':         '64–128 enough for most log grammars',
    'embedding dim':      'same order as units',
    'batch size':         '64–256',
    'epochs':             'EarlyStopping on val_loss patience=5',
    'threshold':          'start 0.5, then sweep on val PR-curve'
}
for k, v in CHEATSHEET.items():
    print(f'{k:20s} : {v}')

---
**You now own the full mental model, the code, the metrics, and the production path.**  
Nothing essential is left to search elsewhere.  
Go build.